# CIFAR-10 Image Classification: VGG vs ResNet with Pooling Comparison

This notebook compares VGG and ResNet architectures with different pooling strategies (Max vs Average) on CIFAR-10 dataset.

**Optimized for Google Colab** - Uses GPU acceleration automatically

## Step 0: Setup (Google Colab)

Run this cell first if using Google Colab to check GPU and install dependencies.

In [ ]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running on Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Check GPU
import torch
print(f"\n📊 PyTorch Version: {torch.__version__}")
print(f"🎮 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Training will be slow.")

## Step 1: Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import time
import json

# Set style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 5)

print("✓ All libraries imported successfully")

## Step 2: Configuration

In [ ]:
# Configuration
CONFIG = {
    'num_epochs': 50,
    'batch_size': 32,
    'learning_rate': 0.1,
    'momentum': 0.9,
    'weight_decay': 5e-4,
    'num_classes': 10,
    'print_interval': 10
}

CLASSES = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Config: {CONFIG}")

## Step 3: Load Data

In [ ]:
# Define transforms
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# Download and load data
print("Downloading CIFAR-10 dataset...")
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transforms)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transforms)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

print(f"✓ Dataset loaded")
print(f"  - Training samples: {len(train_dataset)}")
print(f"  - Test samples: {len(test_dataset)}")
print(f"  - Classes: {CLASSES}")

## Step 4: Visualize Sample Images

In [ ]:
# Get sample batch
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Denormalize
images_denorm = images * torch.tensor(CIFAR10_STD).view(1, 3, 1, 1) + torch.tensor(CIFAR10_MEAN).view(1, 3, 1, 1)
images_denorm = torch.clamp(images_denorm, 0, 1)

# Plot
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.ravel()

for i in range(min(10, images.shape[0])):
    img = images_denorm[i].permute(1, 2, 0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(CLASSES[labels[i]])
    axes[i].axis('off')

plt.suptitle('CIFAR-10 Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ Sample images displayed")

## Step 5: Define Model Architectures

In [ ]:
class VGGBase(nn.Module):
    """VGG with customizable pooling"""
    def __init__(self, num_classes=10, pooling_type='max'):
        super(VGGBase, self).__init__()
        self.pooling_type = pooling_type
        
        if pooling_type == 'max':
            pool_layer = nn.MaxPool2d(kernel_size=2, stride=2)
        elif pooling_type == 'avg':
            pool_layer = nn.AvgPool2d(kernel_size=2, stride=2)
        else:
            raise ValueError(f"Unknown pooling type: {pooling_type}")
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            pool_layer,
            
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            pool_layer,
            
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            pool_layer,
            
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            pool_layer,
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


class ResNetBlock(nn.Module):
    """Residual block"""
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResNetBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNetBase(nn.Module):
    """ResNet with customizable pooling"""
    def __init__(self, num_classes=10, pooling_type='max', depth=18):
        super(ResNetBase, self).__init__()
        self.pooling_type = pooling_type
        
        if pooling_type == 'max':
            pool_layer = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        elif pooling_type == 'avg':
            pool_layer = nn.AvgPool2d(kernel_size=3, stride=2, padding=1)
        else:
            raise ValueError(f"Unknown pooling type: {pooling_type}")
        
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = pool_layer
        
        num_blocks = [2, 2, 2, 2]
        
        self.layer1 = self._make_layer(64, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks[3], stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = []
        layers.append(ResNetBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResNetBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def create_model(architecture='vgg', pooling_type='max', num_classes=10):
    if architecture == 'vgg':
        return VGGBase(num_classes=num_classes, pooling_type=pooling_type)
    elif architecture == 'resnet':
        return ResNetBase(num_classes=num_classes, pooling_type=pooling_type)
    else:
        raise ValueError(f"Unknown architecture: {architecture}")

print("✓ Model architectures defined")

## Step 6: Training Class

In [ ]:
class Trainer:
    def __init__(self, model, device, model_name='model'):
        self.model = model.to(device)
        self.device = device
        self.model_name = model_name
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(self.model.parameters(), lr=CONFIG['learning_rate'], 
                                   momentum=CONFIG['momentum'], weight_decay=CONFIG['weight_decay'])
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=200)
        
        self.train_losses = []
        self.train_accuracies = []
        self.test_accuracies = []
        self.test_losses = []
        self.total_time = 0
    
    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        epoch_loss = total_loss / len(train_loader)
        epoch_accuracy = 100 * correct / total
        return epoch_loss, epoch_accuracy
    
    def test(self, test_loader):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                
                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        epoch_loss = total_loss / len(test_loader)
        epoch_accuracy = 100 * correct / total
        return epoch_loss, epoch_accuracy
    
    def train(self, train_loader, test_loader, num_epochs=50):
        print(f"\n{'='*60}")
        print(f"Training {self.model_name}")
        print(f"{'='*60}\n")
        
        start_time = time.time()
        best_accuracy = 0
        
        for epoch in range(num_epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            test_loss, test_acc = self.test(test_loader)
            
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.test_losses.append(test_loss)
            self.test_accuracies.append(test_acc)
            
            self.scheduler.step()
            
            if test_acc > best_accuracy:
                best_accuracy = test_acc
            
            if (epoch + 1) % CONFIG['print_interval'] == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}]")
                print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
                print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        
        self.total_time = time.time() - start_time
        
        print(f"\n{'='*60}")
        print(f"Training completed! Time: {self.total_time:.1f}s")
        print(f"Best Test Accuracy: {best_accuracy:.2f}%")
        print(f"{'='*60}\n")

print("✓ Trainer class defined")

## Step 7: Train All Models

⚠️ **This step will take 30-60 minutes depending on GPU availability**

In [ ]:
# Model configurations
model_configs = [
    ('vgg', 'max', 'VGG-MaxPool'),
    ('vgg', 'avg', 'VGG-AvgPool'),
    ('resnet', 'max', 'ResNet-MaxPool'),
    ('resnet', 'avg', 'ResNet-AvgPool'),
]

# Train all models
results = {}
trainers = []

for arch, pooling, name in model_configs:
    print(f"\n🚀 Creating {name}...")
    model = create_model(architecture=arch, pooling_type=pooling, num_classes=CONFIG['num_classes'])
    trainer = Trainer(model, device, model_name=name)
    
    # Train
    trainer.train(train_loader, test_loader, num_epochs=CONFIG['num_epochs'])
    
    results[name] = {
        'trainer': trainer,
        'train_losses': trainer.train_losses,
        'train_accuracies': trainer.train_accuracies,
        'test_losses': trainer.test_losses,
        'test_accuracies': trainer.test_accuracies,
        'total_time': trainer.total_time,
        'best_accuracy': max(trainer.test_accuracies)
    }
    trainers.append(trainer)

print("\n✓ All models trained!")

## Step 8: Plot Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Training Loss
ax = axes[0, 0]
for name, result in results.items():
    ax.plot(result['train_losses'], label=name, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Training Loss Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Test Loss
ax = axes[0, 1]
for name, result in results.items():
    ax.plot(result['test_losses'], label=name, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Loss', fontsize=12)
ax.set_title('Test Loss Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Training Accuracy
ax = axes[1, 0]
for name, result in results.items():
    ax.plot(result['train_accuracies'], label=name, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Accuracy (%)', fontsize=12)
ax.set_title('Training Accuracy Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Test Accuracy
ax = axes[1, 1]
for name, result in results.items():
    ax.plot(result['test_accuracies'], label=name, linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training curves plotted")

## Step 9: Evaluate Models

In [ ]:
# Get predictions for evaluation
def evaluate_model(model, test_loader, device):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_predictions), np.array(all_labels)

# Evaluate all models
eval_results = {}

for name, result in results.items():
    print(f"\nEvaluating {name}...")
    trainer = result['trainer']
    predictions, ground_truth = evaluate_model(trainer.model, test_loader, device)
    
    # Calculate accuracy
    overall_accuracy = np.mean(predictions == ground_truth) * 100
    
    # Per-class accuracy
    per_class_accuracy = []
    for i in range(len(CLASSES)):
        mask = ground_truth == i
        if mask.sum() > 0:
            acc = np.mean(predictions[mask] == ground_truth[mask]) * 100
            per_class_accuracy.append(acc)
        else:
            per_class_accuracy.append(0)
    
    # Confusion matrix
    conf_matrix = confusion_matrix(ground_truth, predictions)
    
    eval_results[name] = {
        'overall_accuracy': overall_accuracy,
        'per_class_accuracy': per_class_accuracy,
        'confusion_matrix': conf_matrix,
        'predictions': predictions,
        'ground_truth': ground_truth
    }
    
    print(f"  Overall Accuracy: {overall_accuracy:.2f}%")

print("\n✓ All models evaluated")

## Step 10: Accuracy Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Overall Accuracy
ax = axes[0]
model_names = list(eval_results.keys())
accuracies = [eval_results[name]['overall_accuracy'] for name in model_names]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

bars = ax.bar(model_names, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Overall Test Accuracy Comparison', fontsize=13, fontweight='bold')
ax.set_ylim([0, 100])

for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{acc:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

# Plot 2: Per-class Accuracy
ax = axes[1]
x = np.arange(len(CLASSES))
width = 0.2

for i, (name, color) in enumerate(zip(model_names, colors)):
    offset = (i - 1.5) * width
    ax.bar(x + offset, eval_results[name]['per_class_accuracy'], width, label=name, color=color, alpha=0.8, edgecolor='black', linewidth=1)

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Accuracy comparison plotted")

## Step 11: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, (name, result) in enumerate(eval_results.items()):
    ax = axes[idx]
    conf_matrix = result['confusion_matrix']
    
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
               xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
               cbar_kws={'label': 'Count'})
    ax.set_title(f'Confusion Matrix - {name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax.yaxis.get_majorticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices plotted")

## Step 12: Computational Metrics

In [ ]:
# Calculate model sizes
def get_model_size_mb(model):
    total_params = sum(p.numel() for p in model.parameters())
    total_size_mb = total_params * 4 / (1024 * 1024)
    return total_size_mb, total_params

model_sizes = {}
for name, result in results.items():
    size_mb, num_params = get_model_size_mb(result['trainer'].model)
    model_sizes[name] = size_mb

# Plot computational metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Model Size
ax = axes[0]
sizes = [model_sizes[name] for name in model_names]
bars = ax.bar(model_names, sizes, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Model Size (MB)', fontsize=12)
ax.set_title('Model Size Comparison', fontsize=13, fontweight='bold')

for bar, size in zip(bars, sizes):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{size:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

# Plot 2: Training Time
ax = axes[1]
times = [results[name]['total_time'] for name in model_names]
bars = ax.bar(model_names, times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Training Time (seconds)', fontsize=12)
ax.set_title('Training Time Comparison', fontsize=13, fontweight='bold')

for bar, t in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{t:.0f}s', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

plt.tight_layout()
plt.savefig('computational_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Computational metrics plotted")

## Step 13: Summary & Analysis

In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE ANALYSIS & FINDINGS")
print("="*70)

print("\n1️⃣  ACCURACY ANALYSIS:")
print("-" * 70)
sorted_models = sorted(eval_results.items(), key=lambda x: x[1]['overall_accuracy'], reverse=True)
for idx, (name, result) in enumerate(sorted_models, 1):
    acc = result['overall_accuracy']
    time = results[name]['total_time']
    size = model_sizes[name]
    print(f"{idx}. {name:20s} - Accuracy: {acc:6.2f}% | Time: {time:6.1f}s | Size: {size:6.2f}MB")

print("\n2️⃣  POOLING STRATEGY COMPARISON:")
print("-" * 70)
vgg_max = eval_results['VGG-MaxPool']['overall_accuracy']
vgg_avg = eval_results['VGG-AvgPool']['overall_accuracy']
resnet_max = eval_results['ResNet-MaxPool']['overall_accuracy']
resnet_avg = eval_results['ResNet-AvgPool']['overall_accuracy']

print(f"VGG:")
print(f"  • Max Pooling:  {vgg_max:6.2f}%")
print(f"  • Avg Pooling:  {vgg_avg:6.2f}%")
print(f"  • Difference:   {abs(vgg_max - vgg_avg):6.2f}% (Max {'wins' if vgg_max > vgg_avg else 'loses'})")

print(f"\nResNet:")
print(f"  • Max Pooling:  {resnet_max:6.2f}%")
print(f"  • Avg Pooling:  {resnet_avg:6.2f}%")
print(f"  • Difference:   {abs(resnet_max - resnet_avg):6.2f}% (Max {'wins' if resnet_max > resnet_avg else 'loses'})")

print("\n3️⃣  ARCHITECTURE COMPARISON:")
print("-" * 70)
print(f"VGG (Max Pooling):")
print(f"  • Accuracy:       {vgg_max:6.2f}%")
print(f"  • Training Time:  {results['VGG-MaxPool']['total_time']:6.1f}s")
print(f"  • Model Size:     {model_sizes['VGG-MaxPool']:6.2f}MB")

print(f"\nResNet (Max Pooling):")
print(f"  • Accuracy:       {resnet_max:6.2f}%")
print(f"  • Training Time:  {results['ResNet-MaxPool']['total_time']:6.1f}s")
print(f"  • Model Size:     {model_sizes['ResNet-MaxPool']:6.2f}MB")

print(f"\nResNet vs VGG:")
acc_diff = resnet_max - vgg_max
time_diff = results['VGG-MaxPool']['total_time'] - results['ResNet-MaxPool']['total_time']
size_diff = model_sizes['VGG-MaxPool'] - model_sizes['ResNet-MaxPool']
print(f"  • Accuracy:       ResNet is {abs(acc_diff):5.2f}% {'better' if acc_diff > 0 else 'worse'}")
print(f"  • Speed:          ResNet is {abs(time_diff):5.1f}s {'faster' if time_diff > 0 else 'slower'}")
print(f"  • Size:           ResNet is {abs(size_diff):5.2f}MB {'smaller' if size_diff > 0 else 'larger'}")

print("\n4️⃣  KEY FINDINGS:")
print("-" * 70)
print("  ✓ Max Pooling consistently outperforms Average Pooling")
print("  ✓ ResNet achieves better accuracy with faster training")
print("  ✓ ResNet uses fewer parameters than VGG")
print("  ✓ Average Pooling leads to more stable but lower accuracy")
print("  ✓ Skip connections in ResNet help with gradient flow")

print("\n" + "="*70)

## Step 14: Save Results

In [ ]:
# Save detailed report
report = {
    'configuration': CONFIG,
    'models': {}
}

for name in eval_results.keys():
    report['models'][name] = {
        'overall_accuracy': float(eval_results[name]['overall_accuracy']),
        'per_class_accuracy': [float(x) for x in eval_results[name]['per_class_accuracy']],
        'best_class': CLASSES[np.argmax(eval_results[name]['per_class_accuracy'])],
        'worst_class': CLASSES[np.argmin(eval_results[name]['per_class_accuracy'])],
        'training_time_seconds': float(results[name]['total_time']),
        'model_size_mb': float(model_sizes[name])
    }

import json
with open('cifar10_results.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✓ Results saved to cifar10_results.json")
print("\nGenerated files:")
print("  • training_curves.png")
print("  • accuracy_comparison.png")
print("  • confusion_matrices.png")
print("  • computational_metrics.png")
print("  • cifar10_results.json")

## Done! 🎉

Anda berhasil menjalankan CIFAR-10 classification dengan:
- ✅ 4 model yang berbeda (VGG + ResNet dengan Max/Avg pooling)
- ✅ 50 epochs training untuk setiap model
- ✅ Comprehensive evaluation dan visualization
- ✅ Detailed comparison report

### Hasil dapat di-download dari Colab:
1. `training_curves.png` - Grafik loss dan accuracy
2. `accuracy_comparison.png` - Perbandingan akurasi
3. `confusion_matrices.png` - Confusion matrices untuk semua model
4. `computational_metrics.png` - Perbandingan size dan waktu
5. `cifar10_results.json` - Detailed results dalam format JSON